# STAGM: Spatial-Transcriptomics Analysis via Graph-Mamba
---
## Overview
This notebook presents the implementation of **STAGM** (Spatial-Transcriptomics Analysis via Graph-Mamba), a deep learning framework designed to integrate transcriptomic data with image embeddings. By leveraging **Graph Neural Networks (GNNs)** combined with the **Mamba State Space Model (SSM)**, STAGM captures both local spatial interactions and global tissue dependencies to perform robust spatial domain clustering.

### Workflow Summary
1.  **Data Preparation**: Load transcriptomic data, select highly variable genes, and construct a spatial interaction graph.
2.  **Model Construction**: Implement the STAGM architecture (Encoder with GPSConv + Mamba layers) and contrastive learning objectives.
3.  **Experiment Execution**: Train the model, evaluate embeddings, perform clustering, and visualize results.


## Table of Contents
1. [Metodology](#1.-Metodology)
2. [Notebook Preparation](#2.-Notebook-preparation)
3. [STAGM Model Construction](#3.-STAGM-model-construction)
    - [Data Loading & Preprocessing](#3.1-Data-Loading-Preprocessing)
    - [Architecture & Components](#3.2-Architecture-Components)
4. [Run an Experiment](#4.-Run-an-experiment)


---
<a name="1.-Metodology"></a>
## 1. Metodology
### Hypothesis
Spatial transcriptomics data can be accurately clustered into distinct biological domains by integrating **spatial graph topology** with **global context modeling**.

### Premise
Standard clustering methods often treat spots as independent entities or rely solely on local neighborhood information. STAGM posits that:
1.  **Local Interactions**: Spots interact primarily with their immediate spatial neighbors (modeled via Graph Convolutional Networks).
2.  **Global Context**: Tissue organization exhibits long-range dependencies that can be captured by State Space Models (SSMs).
3.  **Model scalability:** STAGM employs Mamba, which is a computationally efficient and scalable SSM for large graphs.

### Objective
To validate this hypothesis, we aim to demonstrate  or similar clustering accuracy (measured by ARI and NMI) and separation of biological domains compared to baseline methods at a lower computational cost with faster inference time.


---
<a name="2.-Notebook-preparation"></a>
## 2. Notebook Preparation
This section initializes the computational environment, ensures code freshness for dynamic imports, and loads the necessary libraries for data handling (ScanPy, Pandas), linear algebra (NumPy, SciPy), and deep learning (PyTorch, Mamba-SSM, PyTorch Geometric).


In [ ]:
# Makes sure that the latest version of the code is always used when running the notebook
%load_ext autoreload
%autoreload 2

In [ ]:
# Imports used in the notebook
import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from anndata import AnnData
from mamba_ssm import Mamba

In [ ]:
# Add parent directory to path
os.chdir("/workspace/src")

---
<a name="3.-STAGM-model-construction"></a>
## 3. STAGM Model Construction


### 3.1 Data Loading and Preprocessing Pipeline
The `LoadSingle10xAdata` class encapsulates the end-to-end pipeline for preparing spatial transcriptomics data. It performs the following critical steps:
1.  **Data Preprocessing**: Loads 10x Genomics Visium data into an `AnnData` object, preserving spatial coordinates and high-resolution images.
2.  **Feature Selection**: Implements flexible gene selection strategies (`seurat_v3`, `mvp`, `geneclust`, `wgcna`, `spatialde`) to identify highly variable genes.
3.  **Graph Construction**: Builds a $k$-nearest neighbor ($k$-NN) adjacency matrix based on Euclidean distance to define local spot interactions.
4.  **Feature Engineering**:
    *   **Gene Expression**: Processes expression matrices (normalization, log-transformation, scaling).
    *   **Image Embeddings**: Optionally loads and processes visual embeddings, concatenating them with gene features to form a multimodal representation.
5.  **Edge Weighting**: Computes edge probabilities using various kernels (Euclidean, RBF, Cosine) to modulate the strength of connections, reflecting the likelihood of biological interaction.
5.  **Labeling**: Integrates ground truth labels for evaluation when available.

In [ ]:
# Load necessary libraries for gene selection
import ot
from scipy.sparse.csc import csc_matrix
from scipy.sparse.csr import csr_matrix
from scipy.spatial.distance import cdist, cosine, euclidean
from scipy.special import softmax
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

In [ ]:

class LoadSingle10xAdata:
    """
    Load and preprocess 10x Genomics Visium spatial transcriptomics data in single sample format.
    
    Args:
        path:           Path to the directory containing the slide.
        n_top_genes:    Number of top variable genes to select for downstream analysis.
        n_neighbors:    Number of neighbors to consider for constructing the interaction graph.
        image_emb:      Whether the dataset includes image embeddings.
        label:          Whether the dataset includes cell type labels.
        filter_na:      Whether to filter out spots with missing values.
        select:         Method for selecting highly variable genes. Options include "default", "mvp", "geneclust", "wgcna", and "spatialde".
        
    Returns:
        None
    """
    def __init__(
        self,
        path: str,
        n_top_genes: int = 3000,
        n_neighbors: int = 3,
        image_emb: bool = False,
        label: bool = True,
        filter_na: bool = True,
        select: str = "default",
    ) -> None:
        self.path = path
        self.n_top_genes = n_top_genes
        self.n_neighbors = n_neighbors
        self.adata = None
        self.image_emb = image_emb
        self.label = label
        self.filter_na = filter_na
        self.kernel = "euclidean"
        self.select = "default"


    def load_data(self) -> None:
        self.adata = sc.read_visium(
            self.path, count_file="filtered_feature_bc_matrix.h5", load_images=True
        )
        self.adata.var_names_make_unique()


    def preprocess(self) -> None:
        if self.select == "default":
            sc.pp.highly_variable_genes(
                self.adata, flavor="seurat_v3", n_top_genes=self.n_top_genes
            )
            sc.pp.normalize_total(self.adata, target_sum=1e4)
            sc.pp.log1p(self.adata)
            sc.pp.scale(self.adata, zero_center=False, max_value=10)
            
        if self.select == "mvp":
            sc.pp.highly_variable_genes(self.adata, flavor="seurat")
            sc.pp.normalize_total(self.adata, target_sum=1e4)
            sc.pp.log1p(self.adata)
            sc.pp.scale(self.adata, zero_center=False, max_value=10)
            
        if self.select == "geneclust":
            self.adata.X = self.adata.X.toarray()
            info, selected_genes_ps = gc.scGeneClust(
                self.adata, n_var_clusters=200, version="fast", return_info=True
            )
            top_variable_genes = selected_genes_ps.tolist()
            self.adata.var["highly_variable"] = self.adata.var_names.isin(
                top_variable_genes
            )
            sc.pp.normalize_total(self.adata, target_sum=1e4)
            sc.pp.log1p(self.adata)
            sc.pp.scale(self.adata, zero_center=False, max_value=10)
            
        if self.select == "wgcna":
            pyWGCNA_data = PyWGCNA.WGCNA(
                name="data",
                species="human",
                anndata=self.adata,
                outputPath="",
                save=True,
            )
            pyWGCNA_data.preprocess()
            pyWGCNA_data.findModules()
            module_colors = np.unique(pyWGCNA_data.datExpr.var["moduleColors"]).tolist()
            all_hub_genes = []
            for module_color in module_colors:
                df_hub_genes = pyWGCNA_data.top_n_hub_genes(
                    moduleName=module_color, n=100
                )
                gene_names = df_hub_genes.index.tolist()
                all_hub_genes.extend(gene_names)
            self.adata.var["highly_variable"] = self.adata.var_names.isin(all_hub_genes)
            sc.pp.normalize_total(self.adata, target_sum=1e4)
            sc.pp.log1p(self.adata)
            sc.pp.scale(self.adata, zero_center=False, max_value=10)


        if self.select == "spatialde":
            x_coords = self.adata.obsm["spatial"][:, 0]
            y_coords = self.adata.obsm["spatial"][:, 1]
            counts = pd.DataFrame(
                self.adata.X.toarray(),
                index=self.adata.obs_names,
                columns=self.adata.var_names,
            )
            counts = counts.T[counts.sum(0) >= 3].T
            self.adata.obs["total_counts"] = np.ravel(self.adata.X.sum(axis=1))
            sample_info = pd.DataFrame(
                {
                    "x": x_coords,
                    "y": y_coords,
                    "total_counts": self.adata.obs["total_counts"],
                },
                index=self.adata.obs_names,
            )
            norm_expr = NaiveDE.stabilize(counts.T).T
            resid_expr = NaiveDE.regress_out(
                sample_info, norm_expr.T, "np.log(total_counts)"
            ).T
            sample_resid_expr = resid_expr.sample(n=1000, axis=1, random_state=1)
            X = sample_info[["x", "y"]].values
            results = SpatialDE.run(X, resid_expr)
            top_genes_list = results.sort_values("qval")["g"].head(1000).tolist()
            self.adata.var["highly_variable"] = self.adata.var_names.isin(
                top_genes_list
            )
            sc.pp.normalize_total(self.adata, target_sum=1e4)
            sc.pp.log1p(self.adata)
            sc.pp.scale(self.adata, zero_center=False, max_value=10)


    def construct_interaction(self) -> None:
        position = self.adata.obsm["spatial"]
        distance_matrix = ot.dist(position, position, metric="euclidean")
        n_spot = distance_matrix.shape[0]
        interaction = np.zeros([n_spot, n_spot])
        for i in range(n_spot):
            vec = distance_matrix[i, :]
            distance = vec.argsort()
            for t in range(1, self.n_neighbors + 1):
                y = distance[t]
                interaction[i, y] = 1

        adj = interaction + interaction.T
        adj = np.where(adj > 1, 1, adj)

        self.adata.obsm["graph_neigh"] = adj


    def generate_gene_expr(self) -> None:
        adata_Vars = self.adata[:, self.adata.var["highly_variable"]]
        if isinstance(adata_Vars.X, csc_matrix) or isinstance(adata_Vars.X, csr_matrix):
            feat = adata_Vars.X.toarray()[:,]
        else:
            feat = adata_Vars.X[:,]

        self.adata.obsm["feat"] = feat


    def load_label(self) -> None:
        df_meta = pd.read_csv(
            os.path.join(self.path, "truth.txt"), sep="\t", header=None
        )
        df_meta_layer = df_meta[1]

        self.adata.obs["ground_truth"] = df_meta_layer.values

        if self.filter_na:
            self.adata = self.adata[~pd.isnull(self.adata.obs["ground_truth"])]


    def load_image_emb(self) -> None:
        data = np.load(os.path.join(self.path, "embeddings.npy"))
        data = data.reshape(data.shape[0], -1)
        scaler = StandardScaler()
        embedding = scaler.fit_transform(data)
        pca = PCA(n_components=16, random_state=42)
        embedding = pca.fit_transform(embedding)
        self.adata.obsm["img_emb"] = embedding
        pca_g = PCA(n_components=64, random_state=42)
        self.adata.obsm["feat_pca"] = pca_g.fit_transform(self.adata.obsm["feat"])
        self.adata.obsm["con_feat"] = np.concatenate(
            [self.adata.obsm["feat_pca"], self.adata.obsm["img_emb"]], axis=1
        )
        con_feat = self.adata.obsm["con_feat"]

        scaler = StandardScaler()

        con_feat_standardized = scaler.fit_transform(con_feat)

        self.adata.obsm["con_feat"] = con_feat_standardized


    def calculate_edge_weights(self) -> None:

        graph_neigh = self.adata.obsm["graph_neigh"]
        node_emb = self.adata.obsm["img_emb"]
        num_nodes = node_emb.shape[0]
        edge_weights = np.zeros_like(graph_neigh)

        for i in tqdm(range(num_nodes), desc="Calculating distances"):
            for j in range(num_nodes):
                if graph_neigh[i, j] == 1:
                    edge_weights[i, j] = euclidean(node_emb[i], node_emb[j])

        edge_probabilities = np.zeros_like(edge_weights)
        for i in tqdm(range(num_nodes), desc="Calculating edge_probabilities"):
            non_zero_indices = edge_weights[i] != 0
            if non_zero_indices.any():
                non_zero_weights = np.log(edge_weights[i][non_zero_indices])
                softmax_weights = softmax(non_zero_weights)
                edge_probabilities[i][non_zero_indices] = softmax_weights

        self.adata.obsm["edge_probabilities"] = edge_probabilities

        if self.kernel == "rbf":
            gamma = 0.01
            similarity_matrix = rbf_kernel(node_emb, gamma=gamma)

            edge_weights = np.where(graph_neigh == 1, 1 - similarity_matrix, 0)

            edge_probabilities = np.zeros_like(edge_weights)
            for i in range(edge_weights.shape[0]):
                non_zero_indices = edge_weights[i] != 0
                non_zero_weights = edge_weights[i][non_zero_indices]
                softmax_weights = softmax(non_zero_weights)
                edge_probabilities[i][non_zero_indices] = softmax_weights

            self.adata.obsm["edge_probabilities"] = edge_probabilities

        if self.kernel == "cosine":
            euclidean_distances = cdist(node_emb, node_emb, metric="cosine")

            edge_weights = np.where(graph_neigh == 1, euclidean_distances, 0)

            edge_probabilities = np.zeros_like(edge_weights)
            for i in range(edge_weights.shape[0]):
                non_zero_indices = edge_weights[i] != 0
                non_zero_weights = edge_weights[i][non_zero_indices]
                softmax_weights = softmax(non_zero_weights)
                edge_probabilities[i][non_zero_indices] = softmax_weights

            self.adata.obsm["edge_probabilities"] = edge_probabilities

    def calculate_edge_weights_gene(self) -> None:

        graph_neigh = self.adata.obsm["graph_neigh"]
        node_emb = self.adata.obsm["feat"]
        scaler = StandardScaler()
        embedding = scaler.fit_transform(node_emb)
        pca = PCA(n_components=64, random_state=42)
        embedding = pca.fit_transform(embedding)
        node_emb = embedding

        num_nodes = node_emb.shape[0]
        edge_weights = np.zeros((num_nodes, num_nodes))

        for i in tqdm(range(num_nodes), desc="Calculating distances"):
            for j in range(num_nodes):
                if graph_neigh[i, j] == 1:
                    edge_weights[i, j] = cosine(node_emb[i], node_emb[j])

        edge_probabilities = np.zeros_like(edge_weights)
        for i in range(num_nodes):
            non_zero_indices = edge_weights[i] != 0
            if non_zero_indices.any():
                non_zero_weights = edge_weights[i][non_zero_indices]
                softmax_weights = softmax(non_zero_weights)
                edge_probabilities[i][non_zero_indices] = softmax_weights

        self.adata.obsm["edge_probabilities"] = edge_probabilities

    def run(self) -> AnnData:
        self.load_data()
        if self.label:
            self.load_label()
        self.preprocess()
        self.construct_interaction()
        self.generate_gene_expr()

        if self.image_emb:
            self.load_image_emb()
            self.calculate_edge_weights()
        else:
            self.calculate_edge_weights_gene()

        print("adata load done")

        return self.adata

### 3.2 The STAGM Wrapper Class
The `STAGM` class manages the interaction between data, the neural network architecture, and the training loop.
*   **Initialization**: Instantiates the model based on configuration parameters.
    *   In **MV mode**, it utilizes image-derived pseudo-labels to guide the contrastive learning objective.
    *   In **SV mode**, it relies solely on graph structure and gene expression without external pseudo-supervision.

*   **Training Loop**:
    *   **Augmentation**: Generates two stochastic views of the graph and features using edge dropout and feature dropping to simulate data variation.
    *   **Optimization**: Computes the neighbor-aware contrastive loss, which maximizes similarity between augmented views of the same node while minimizing similarity with neighbors of different classes (in biased mode).

*   **Evaluation**: Extracts the final learned embeddings, reports model diagnostics (parameter counts, memory usage), and validates the embedding distribution.

*   **Visualization**: Provides built-in methods to plot spatial embeddings (UMAP), batch effects, and spatial domain maps on tissue images.


In [ ]:
import argparse
from time import perf_counter

import torch.nn as nn
import tqdm
from sklearn.cluster import KMeans
from torch_geometric.nn import GCNConv


def generate_pseudo_labels(img_emb: np.ndarray, n_clusters: int = 300) -> torch.Tensor:
    kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(img_emb)
    pseudo_labels = kmeans.labels_
    return torch.tensor(pseudo_labels)


def adj_to_edge_index(adj: torch.Tensor) -> torch.Tensor:
    row, col = torch.where(adj != 0)
    edge_index = torch.stack([row, col], dim=0)
    return edge_index


def convert_edge_probabilities(
    adj_matrix: torch.Tensor, edge_prob_matrix: torch.Tensor
) -> torch.Tensor:
    row, col = torch.where(adj_matrix != 0)
    edge_probs = edge_prob_matrix[row, col]
    return edge_probs


class STAGM:
    def __init__(
        self,
        args: argparse.Namespace,
        config: dict,
        single: bool = False,
        refine: bool = True,
    ) -> None:
        self.args: argparse.Namespace = args
        self.single: bool = single
        self.config: dict = config
        self.learning_rate: float = config.learning_rate
        self.num_hidden: int = config.num_hidden
        self.num_proj_hidden: int = config.num_proj_hidden
        self.activation = ({"relu": nn.functional.relu, "prelu": nn.PReLU()})[
            config.activation
        ]
        self.base_model = ({"GCNConv": GCNConv})[config.base_model]
        self.num_layers: int = config.num_layers
        self.drop_feature_rate_1: float = config.drop_feature_rate_1
        self.drop_feature_rate_2: float = config.drop_feature_rate_2
        self.tau: float = config.tau
        self.num_epochs: int = config.num_epochs
        self.weight_decay: float = config.weight_decay
        self.num_clusters: int = config.num_clusters
        self.num_gene: int = config.num_gene
        self.refine: bool = refine
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.radius: int = 15
        self.tool: str = "leiden"  # mclust, leiden, and louvain
        self.bar_format: str = (
            "{l_bar}{bar}| [{elapsed}<{remaining}, {rate_fmt}{postfix}]"
        )
        self.encoder = Encoder(
            self.num_gene,
            self.num_hidden,
            self.activation,
            base_model=self.base_model,
            num_layers=self.num_layers,
            dropout=self.config.dropout,
            order_by_degree=self.config.order_by_degree,
            shuffle_ind=self.config.shuffle_ind,
            d_state=self.config.d_state,
            d_conv=self.config.d_conv,
        ).to(self.device)
        self.adata = None
        self.mask_slices = True

        if single:
            self.model = SVmodel(
                self.encoder, self.num_hidden, self.num_proj_hidden, self.tau
            ).to(self.device)

        else:
            self.model = MVmodel(
                self.encoder, self.num_hidden, self.num_proj_hidden, self.tau
            ).to(self.device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=self.learning_rate,
            weight_decay=self.weight_decay,
        )

    def train(self) -> None:
        print("=== prepare for training ===")
        if self.adata is None:
            raise ValueError("adata not load!")

        features_matrix = torch.FloatTensor(self.adata.obsm["feat"].copy()).to(
            self.device
        )

        graph_neigh = torch.FloatTensor(self.adata.obsm["graph_neigh"].copy()).to(
            self.device
        )
        edge_probabilities = torch.FloatTensor(
            self.adata.obsm["edge_probabilities"].copy()
        ).to(self.device)

        edge_index = adj_to_edge_index(graph_neigh)
        edge_probs = convert_edge_probabilities(graph_neigh, edge_probabilities)
        batch = torch.zeros(
            features_matrix.size(0), dtype=torch.long, device=self.device
        )

        if self.single:
            if ("mask_neigh" in self.adata.obsm) and (self.mask_slices):
                print("Consider intra slice")
                mask_neigh = torch.FloatTensor(self.adata.obsm["mask_neigh"].copy()).to(
                    self.device
                )
            else:
                mask_neigh = None
        else:
            if "pseudo_labels" in self.adata.obs:
                pseudo_labels = torch.tensor(
                    self.adata.obs["pseudo_labels"].cat.codes
                ).to(self.device)
            else:
                if self.config.k:
                    pseudo_labels = generate_pseudo_labels(
                        self.adata.obsm["img_emb"], self.config.k
                    )
                else:
                    pseudo_labels = generate_pseudo_labels(self.adata.obsm["img_emb"])
                pseudo_labels = pseudo_labels.to(self.device)

        self._loss_curve: list[float] = []
        train_start = perf_counter()

        print("=== train ===")
        for _ in tqdm.tqdm(range(1, self.num_epochs + 1), bar_format=self.bar_format):
            self.model.train()
            self.optimizer.zero_grad()
            edge_index_1 = multiple_dropout_average(
                edge_index, edge_probs, force_undirected=True
            )[0]
            edge_index_2 = multiple_dropout_average(
                edge_index, edge_probs, force_undirected=True
            )[0]
            x_1 = drop_feature(features_matrix, self.drop_feature_rate_1)
            x_2 = drop_feature(features_matrix, self.drop_feature_rate_2)
            z1 = self.model(x_1, edge_index_1, batch)
            z2 = self.model(x_2, edge_index_2, batch)

            if self.single:
                loss = self.model.contrastive_loss(z1, z2, graph_neigh, mask=mask_neigh)
            else:
                loss = self.model.contrastive_loss_biased(
                    z1, z2, graph_neigh, pseudo_labels
                )

            loss.backward()
            self.optimizer.step()
            self._loss_curve.append(loss.item())

        self._training_time_seconds: float = perf_counter() - train_start

        if not self.single:
            torch.save(self.model.state_dict(), "model.pt")

    def eva(self):
        print("=== load ===")
        self.model.load_state_dict(torch.load("model.pt"))
        self.model.eval()

        features_matrix = torch.FloatTensor(self.adata.obsm["feat"].copy()).to(
            self.device
        )

        graph_neigh = torch.FloatTensor(self.adata.obsm["graph_neigh"].copy()).to(
            self.device
        )

        edge_index = adj_to_edge_index(graph_neigh)
        batch = torch.zeros(
            features_matrix.size(0), dtype=torch.long, device=self.device
        )

        with torch.no_grad():
            self.adata.obsm["emb"] = (
                self.model(features_matrix, edge_index, batch).detach().cpu().numpy()
            )

        # --- parameter counts ---
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(
            p.numel() for p in self.model.parameters() if p.requires_grad
        )

        # --- static memory footprint (weights + buffers, fp32 equivalent) ---
        param_bytes = sum(p.numel() * p.element_size() for p in self.model.parameters())
        buffer_bytes = sum(b.numel() * b.element_size() for b in self.model.buffers())
        model_mb = (param_bytes + buffer_bytes) / 1024**2

        # --- peak GPU memory allocated during the *current process* ---
        if self.device.type == "cuda":
            peak_gpu_mb = torch.cuda.max_memory_allocated(self.device) / 1024**2
            torch.cuda.reset_peak_memory_stats(self.device)  # reset for next run
        else:
            peak_gpu_mb = None

        # --- training time (set by train(); graceful fallback if eva() called standalone) ---
        training_time = getattr(self, "_training_time_seconds", None)

        # --- loss curve summary ---
        loss_curve = getattr(self, "_loss_curve", [])
        if loss_curve:
            loss_summary = (
                f"  first={loss_curve[0]:.4f}  "
                f"last={loss_curve[-1]:.4f}  "
                f"min={min(loss_curve):.4f}  "
                f"max={max(loss_curve):.4f}"
            )
        else:
            loss_summary = "  (no loss history — train() was not called this session)"

        # --- embedding stats ---
        emb = self.adata.obsm["emb"]
        emb_norm = np.linalg.norm(emb, axis=1)

        print("\n========== Model Diagnostics ==========")
        print(f"  Total parameters:     {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,}")
        print(f"  Model weight size:    {model_mb:.2f} MB")
        if peak_gpu_mb is not None:
            print(f"  Peak GPU memory:      {peak_gpu_mb:.2f} MB")
        if training_time is not None:
            mins, secs = divmod(training_time, 60)
            print(
                f"  Training time:        {int(mins)}m {secs:.1f}s  ({training_time:.1f}s total)"
            )
            if loss_curve:
                print(
                    f"  Time per epoch:       {training_time / len(loss_curve) * 1000:.1f} ms"
                )
        print(f"\n  Loss curve:          {loss_summary}")
        print(f"\n  Embedding shape:      {emb.shape}")
        print(
            f"  Embedding norm — mean={emb_norm.mean():.4f}  "
            f"std={emb_norm.std():.4f}  "
            f"min={emb_norm.min():.4f}  "
            f"max={emb_norm.max():.4f}"
        )
        print("=======================================\n")
        print(self.adata.obsm["emb"])
        print("embedding generated, go clustering")

    def cluster(self, label=True):
        if self.tool == "mclust":
            clustering(
                self.adata,
                self.num_clusters,
                radius=self.radius,
                method=self.tool,
                refinement=self.refine,
            )  # For DLPFC dataset, we use optional refinement step.
        elif self.tool in ["leiden", "louvain"]:
            clustering(
                self.adata,
                self.num_clusters,
                radius=self.radius,
                method=self.tool,
                start=0.01,
                end=0.27,
                increment=0.005,
                refinement=True,
            )

        if label:
            print("calculate metric ARI")
            # calculate metric ARI
            ARI = metrics.adjusted_rand_score(
                self.adata.obs["domain"], self.adata.obs["ground_truth"]
            )
            self.adata.uns["ari"] = ARI
            NMI = metrics.normalized_mutual_info_score(
                self.adata.obs["domain"], self.adata.obs["ground_truth"]
            )
            self.adata.uns["nmi"] = NMI
            print("ARI:", ARI)
            print("NMI:", NMI)
        else:
            print("calculate SC and DB")
            SC = silhouette_score(self.adata.obsm["emb"], self.adata.obs["domain"])
            DB = davies_bouldin_score(self.adata.obsm["emb"], self.adata.obs["domain"])
            self.adata.uns["sc"] = SC
            self.adata.uns["db"] = DB
            print("SC:", SC)
            print("DB:", DB)
        if "batch" in (self.adata.obs):
            BatchKL(self.adata)
            ILISI = hm.compute_lisi(
                self.adata.obsm["emb"],
                self.adata.obs[["batch"]],
                label_colnames=["batch"],
            )[:, 0]
            median_ILISI = np.median(ILISI)
            print(f"Median ILISI: {median_ILISI:.2f}")

    def draw_spatial(self, p=""):
        sc.pl.spatial(
            self.adata,
            img_key="hires",
            size=1.6,
            color=["ground_truth", "domain"],
            show=True,
            save=p + str(self.args.slide) + ".png",
        )

    def draw_single_spatial(self):
        sc.pl.embedding(
            self.adata,
            basis="spatial",
            color="domain",
            size=100,
            save=str(self.args.slide) + ".png",
        )

    def draw_umap(self):
        print("start umap")
        sc.pp.neighbors(self.adata, use_rep="emb")
        sc.tl.umap(self.adata)
        sc.pl.umap(
            self.adata,
            color="domain",
            show=True,
            save=str(self.args.slide) + "domain.pdf",
        )
        sc.pl.umap(
            self.adata,
            color="batch",
            show=True,
            save=str(self.args.slide) + "_batch.pdf",
        )
        if self.args.label == True:
            sc.pl.umap(
                self.adata,
                color="ground_truth",
                show=True,
                save=str(self.args.slide) + "_label.pdf",
            )

    def draw_horizontal(self):
        adata_batch_0 = self.adata[self.adata.obs["batch"] == "0", :]
        sc.pl.embedding(adata_batch_0, basis="spatial", color="domain", save="0.png")

        adata_batch_1 = self.adata[self.adata.obs["batch"] == "1", :]
        sc.pl.embedding(adata_batch_1, basis="spatial", color="domain", save="1.png")


### 3.3 STAGM Architecture Components
The neural backbone of STAGM is a custom **Encoder** that integrates local and global modeling capabilities through `GPSConv` layers.


#### 3.3.1 The GPSConv Layer (Global Pooling + Self-attention)
This layer is the fundamental building block, combining two distinct processing branches:
1.  **Local MPNN Branch**: Utilizes a standard Graph Convolutional Network (`GCNConv`) to aggregate information from immediate spatial neighbors. This captures short-range interactions and local tissue structure.
2.  **Global Mamba Branch**: Employs the **Mamba State Space Model (SSM)** to process node features as a sequence.
    *   **Mechanism**: By sorting nodes (optionally by degree) and applying the Mamba block, the model captures long-range dependencies across the entire tissue section efficiently (linear complexity).
    *   **Integration**: The outputs of the local and global branches are summed and passed through a Feed-Forward Network (FFN), enabling the model to jointly learn local context and global tissue organization.

In [ ]:
import inspect
from typing import Any, Dict, Optional, Tuple

# import torch
from torch import Tensor
from torch.nn import Dropout, Linear, Sequential
from torch_geometric.nn.conv import MessagePassing

# from torch_geometric.nn.inits import reset
from torch_geometric.nn.resolver import activation_resolver, normalization_resolver
from torch_geometric.typing import Adj, OptTensor
from torch_geometric.utils import degree, sort_edge_index, to_dense_batch


def _permute_within_batch(node_features: Tensor, batch: Tensor) -> Tensor:
    """Returns a permuted index tensor that shuffles nodes within each graph."""
    permuted_indices = [
        (batch == b).nonzero().squeeze()[torch.randperm((batch == b).sum().item())]
        for b in torch.unique(batch)
    ]
    return torch.cat(permuted_indices)


class GPSConv(torch.nn.Module):
    """
    GPS-style layer combining a local MPNN with a global Mamba SSM.

    Args:
        channels:         Node feature dimensionality.
        conv:             Local message-passing layer (e.g. GCNConv).
        dropout:          Dropout applied after each sub-layer.
        act:              Activation name for the feed-forward MLP.
        act_kwargs:       Extra kwargs forwarded to the activation resolver.
        norm:             Normalisation layer name (or None).
        norm_kwargs:      Extra kwargs forwarded to the normalisation resolver.
        order_by_degree:  Sort nodes by degree before feeding into Mamba.
        shuffle_ind:      Number of random permutations to average (0 = no shuffle).
        d_state:          Mamba SSM state size.
        d_conv:           Mamba conv kernel size.
    """

    def __init__(
        self,
        channels: int,
        conv: Optional[MessagePassing],
        dropout: float = 0.0,
        act: str = "relu",
        act_kwargs: Optional[Dict[str, Any]] = None,
        norm: Optional[str] = "batch_norm",
        norm_kwargs: Optional[Dict[str, Any]] = None,
        order_by_degree: bool = False,
        shuffle_ind: int = 0,
        d_state: int = 16,
        d_conv: int = 4,
    ):
        super().__init__()

        assert not (order_by_degree and shuffle_ind != 0), (
            f"order_by_degree={order_by_degree} and shuffle_ind={shuffle_ind} "
            "are mutually exclusive"
        )

        self.channels = channels
        self.conv = conv
        self.dropout = dropout
        self.order_by_degree = order_by_degree
        self.shuffle_ind = shuffle_ind

        self.mamba = Mamba(d_model=channels, d_state=d_state, d_conv=d_conv, expand=1)

        self.mlp = Sequential(
            Linear(channels, channels * 2),
            activation_resolver(act, **(act_kwargs or {})),
            Dropout(dropout),
            Linear(channels * 2, channels),
            Dropout(dropout),
        )

        norm_kwargs = norm_kwargs or {}
        self.norm1 = normalization_resolver(norm, channels, **norm_kwargs)
        self.norm2 = normalization_resolver(norm, channels, **norm_kwargs)
        self.norm3 = normalization_resolver(norm, channels, **norm_kwargs)

        self.norm_with_batch = False
        if self.norm1 is not None:
            sig = inspect.signature(self.norm1.forward)
            self.norm_with_batch = "batch" in sig.parameters

    def reset_parameters(self):
        if self.conv is not None:
            self.conv.reset_parameters()
        reset(self.mlp)
        for norm in (self.norm1, self.norm2, self.norm3):
            if norm is not None:
                norm.reset_parameters()

    def _apply_norm(self, norm, node_features: Tensor, batch: Tensor) -> Tensor:
        if norm is None:
            return node_features
        return (
            norm(node_features, batch=batch)
            if self.norm_with_batch
            else norm(node_features)
        )

    def forward(
        self,
        node_features: Tensor,
        edge_index: Adj,
        batch: Tensor,
        **kwargs,
    ) -> Tensor:
        branch_outputs = []

        # --- Local MPNN branch ---
        if self.conv is not None:
            h = self.conv(node_features, edge_index, **kwargs)
            h = nn.functional.dropout(h, p=self.dropout, training=self.training)
            h = h + node_features  # residual
            h = self._apply_norm(self.norm1, h, batch)
            branch_outputs.append(h)

        # --- Global Mamba branch ---
        x = node_features
        if self.order_by_degree:
            deg = degree(edge_index[0], x.size(0)).to(torch.long)
            order_tensor = torch.stack([batch, deg], dim=1).T
            _, x = sort_edge_index(order_tensor, edge_attr=x)

        if self.shuffle_ind == 0:
            dense, mask = to_dense_batch(x, batch)
            h = self.mamba(dense)[mask]
        else:
            shuffled = []
            for _ in range(self.shuffle_ind):
                perm = _permute_within_batch(x, batch)
                dense, mask = to_dense_batch(x[perm], batch)
                h_i = self.mamba(dense)[mask][perm]
                shuffled.append(h_i)
            h = sum(shuffled) / self.shuffle_ind

        h = nn.functional.dropout(h, p=self.dropout, training=self.training)
        h = h + node_features  # residual
        h = self._apply_norm(self.norm2, h, batch)
        branch_outputs.append(h)

        # --- Combine branches + feed-forward ---
        out = sum(branch_outputs)
        out = out + self.mlp(out)
        out = self._apply_norm(self.norm3, out, batch)

        return out

#### 3.3.2 The Encoder
The Encoder stacks multiple `GPSConv` layers to learn hierarchical representations:
*   **Projection**: Maps raw high-dimensional gene features into a latent space.
*   **Residual Connections**: Ensures gradient flow across deep layers.
*   **Output Projection**: Reduces the embedding dimension to the target size (`out_channels`).

In [ ]:
class Encoder(torch.nn.Module):
    """
    Multi-layer graph encoder using GPSConv (local GCNConv + global Mamba).

    The width schedule mirrors the original:
      - 1 layer:  in_channels -> out_channels
      - k layers: in_channels -> 2*out_channels (hidden) -> out_channels

    Args:
        in_channels:      Input feature size.
        out_channels:     Output embedding size.
        activation:       Activation applied after each GPSConv.
        base_model:       Local MPNN constructor (default: GCNConv).
        num_layers:       Total number of GPSConv layers.
        dropout:          Dropout rate inside each GPSConv.
        order_by_degree:  Sort nodes by degree before Mamba.
        shuffle_ind:      Number of random permutation averages (0 = none).
        d_state:          Mamba SSM state size.
        d_conv:           Mamba conv kernel size.
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        activation,
        base_model=GCNConv,
        num_layers: int = 2,
        dropout: float = 0.0,
        order_by_degree: bool = False,
        shuffle_ind: int = 0,
        d_state: int = 16,
        d_conv: int = 4,
    ):
        super().__init__()
        assert num_layers >= 1

        self.num_layers = num_layers
        self.activation = activation

        hidden_channels = out_channels if num_layers == 1 else 2 * out_channels

        # Project raw node features into the working width
        self.input_proj = (
            nn.Identity()
            if in_channels == hidden_channels
            else Linear(in_channels, hidden_channels)
        )

        def _make_gps(in_ch: int, out_ch: int) -> GPSConv:
            return GPSConv(
                channels=in_ch,
                conv=base_model(in_ch, out_ch),
                dropout=dropout,
                order_by_degree=order_by_degree,
                shuffle_ind=shuffle_ind,
                d_state=d_state,
                d_conv=d_conv,
            )

        gps_layers: list[nn.Module] = []

        if num_layers == 1:
            # Single layer: GPSConv input/output both at out_channels
            gps_layers.append(_make_gps(out_channels, out_channels))
        else:
            # Hidden layers stay at 2*out_channels
            for _ in range(num_layers - 1):
                gps_layers.append(_make_gps(2 * out_channels, 2 * out_channels))
            # Final layer: local conv narrows to out_channels;
            # GPSConv still outputs `channels` (= 2*out_channels) so we
            # add a linear readout to squeeze to out_channels.
            gps_layers.append(_make_gps(2 * out_channels, out_channels))
            self.output_proj = Linear(2 * out_channels, out_channels)

        self.gps_layers = nn.ModuleList(gps_layers)
        self._needs_output_proj = num_layers > 1

    def forward(
        self,
        node_features: Tensor,
        edge_index: Tensor,
        batch: Tensor,
    ) -> Tensor:
        """
        Args:
            node_features: (N, in_channels)
            edge_index:    (2, E)
            batch:         (N,)  graph-assignment vector
        Returns:
            Node embeddings of shape (N, out_channels)
        """
        node_emb = self.input_proj(node_features)

        for layer in self.gps_layers:
            node_emb = self.activation(layer(node_emb, edge_index, batch))

        if self._needs_output_proj:
            node_emb = self.output_proj(node_emb)

        return node_emb

#### 3.3.3 Contrastive Learning Heads
The model includes `MVmodel` and `SVmodel` classes which implement specific contrastive objectives:
*   **Projection Head**: A two-layer MLP with ELU activation to project the encoder output into a space suitable for contrastive comparison.
*   **Neighbor-Aware Loss**: Modifies the standard NT-Xent loss by incorporating the adjacency matrix, ensuring that positive pairs are not only self-matches but also spatially connected neighbors.
*   **Biased Loss (MV)**: Introduces a masking mechanism based on pseudo-labels, treating nodes from different pseudo-clusters as stronger negatives, thereby sharpening domain boundaries.

In [ ]:
class _ProjectionMixin(torch.nn.Module):
    """Shared projection head and cosine-similarity helpers for MV/SV models."""

    def __init__(self, num_hidden: int, num_proj_hidden: int, tau: float):
        super().__init__()
        self.tau = tau
        self.forward_proj_1 = nn.Linear(num_hidden, num_proj_hidden)
        self.forward_proj_2 = nn.Linear(num_proj_hidden, num_hidden)

    def projection(self, gnn_embedding: Tensor) -> Tensor:
        projected = nn.functional.elu(self.forward_proj_1(gnn_embedding))
        return self.forward_proj_2(projected)

    def similarity_matrix(self, emb_a: Tensor, emb_b: Tensor) -> Tensor:
        """Normalised dot-product similarity matrix."""
        return torch.mm(
            nn.functional.normalize(emb_a), nn.functional.normalize(emb_b).t()
        )

    def tau_scaling(self, sim: Tensor) -> Tensor:
        """Temperature-scaled exponential (NT-Xent numerator/denominator)."""
        return torch.exp(sim / self.tau)

    def _reduce(self, per_node_loss: Tensor, mean: bool) -> Tensor:
        return per_node_loss.mean() if mean else per_node_loss.sum()

    @staticmethod
    def _strip_self_loops(adj: Tensor) -> Tensor:
        adj = adj - torch.diag_embed(adj.diag())
        adj[adj > 0] = 1
        return adj

    @staticmethod
    def _positive_pair_counts(adj: Tensor) -> Tensor:
        """2 * |N_i| + 1  (intra + inter neighbours + self inter-view)."""
        return torch.sum(adj, 1).mul(2).add(1).squeeze()

In [ ]:
class MVmodel(_ProjectionMixin):
    """
    Multi-view contrastive model.
    `batch` is now a required argument because the Mamba encoder needs it.
    """

    def __init__(
        self,
        encoder: Encoder,
        num_hidden: int,
        num_proj_hidden: int,
        tau: float = 0.5,
    ):
        super().__init__(num_hidden, num_proj_hidden, tau)
        self.encoder = encoder

    def forward(
        self, node_features: Tensor, edge_index: Tensor, batch: Tensor
    ) -> Tensor:
        gnn_embedding = self.encoder(node_features, edge_index, batch)
        projected_embedding = self.projection(gnn_embedding)
        return projected_embedding

    # -- Basic (non-neighbour-aware) contrastive loss -----------------------

    def _semi_loss(self, emb_a: Tensor, emb_b: Tensor) -> Tensor:
        self_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_a))
        cross_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_b))
        return -torch.log(
            cross_sim.diag() / (self_sim.sum(1) + cross_sim.sum(1) - self_sim.diag())
        )

    def loss(
        self,
        proj_emb_1: Tensor,
        proj_emb_2: Tensor,
        mean: bool = True,
        batch_size: int = 0,
    ) -> Tensor:
        per_node = (
            self._semi_loss(proj_emb_1, proj_emb_2)
            + self._semi_loss(proj_emb_2, proj_emb_1)
        ) * 0.5
        return self._reduce(per_node, mean)

    # -- Neighbour-aware contrastive loss -----------------------------------

    def _neighbor_contrastive_loss(
        self, emb_a: Tensor, emb_b: Tensor, adj: Tensor
    ) -> Tensor:
        adj = self._strip_self_loops(adj)
        positive_pair_counts = self._positive_pair_counts(adj)

        intra_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_a))
        inter_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_b))

        numerator = (
            inter_sim.diag() + intra_sim.mul(adj).sum(1) + inter_sim.mul(adj).sum(1)
        )
        denominator = intra_sim.sum(1) + inter_sim.sum(1) - intra_sim.diag()

        return -torch.log((numerator / denominator) / positive_pair_counts)

    def contrastive_loss(
        self, emb_a: Tensor, emb_b: Tensor, adj: Tensor, mean: bool = True
    ) -> Tensor:
        per_node = (
            self._neighbor_contrastive_loss(emb_a, emb_b, adj)
            + self._neighbor_contrastive_loss(emb_b, emb_a, adj)
        ) * 0.5
        return self._reduce(per_node, mean)

    # -- Biased neighbour-aware contrastive loss ----------------------------

    def _neighbor_contrastive_loss_biased(
        self, emb_a: Tensor, emb_b: Tensor, adj: Tensor, pseudo_labels: Tensor
    ) -> Tensor:
        adj = self._strip_self_loops(adj)
        positive_pair_counts = self._positive_pair_counts(adj)

        intra_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_a))
        inter_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_b))

        negative_mask = (pseudo_labels.view(-1, 1) != pseudo_labels.view(1, -1)).float()
        masked_intra_sim = intra_sim * negative_mask
        masked_inter_sim = inter_sim * negative_mask

        numerator = (
            inter_sim.diag() + intra_sim.mul(adj).sum(1) + inter_sim.mul(adj).sum(1)
        )
        denominator = (
            masked_intra_sim.sum(1) + masked_inter_sim.sum(1) - intra_sim.diag()
        )

        return -torch.log((numerator / denominator) / positive_pair_counts)

    def contrastive_loss_biased(
        self,
        emb_a: Tensor,
        emb_b: Tensor,
        adj: Tensor,
        pseudo_labels: Tensor,
        mean: bool = True,
    ) -> Tensor:
        per_node = (
            self._neighbor_contrastive_loss_biased(emb_a, emb_b, adj, pseudo_labels)
            + self._neighbor_contrastive_loss_biased(emb_b, emb_a, adj, pseudo_labels)
        ) * 0.5
        return self._reduce(per_node, mean)

In [ ]:
class SVmodel(_ProjectionMixin):
    """
    Single-view contrastive model.
    `batch` is now a required argument because the Mamba encoder needs it.
    """

    def __init__(
        self,
        encoder: Encoder,
        num_hidden: int,
        num_proj_hidden: int,
        tau: float = 0.5,
    ):
        super().__init__(num_hidden, num_proj_hidden, tau)
        self.encoder = encoder

    def forward(
        self, node_features: Tensor, edge_index: Tensor, batch: Tensor
    ) -> Tensor:
        gnn_embedding = self.encoder(node_features, edge_index, batch)
        projected_embedding = self.projection(gnn_embedding)
        return projected_embedding

    def _neighbor_contrastive_loss(
        self,
        emb_a: Tensor,
        emb_b: Tensor,
        adj: Tensor,
        sample_mask: Optional[Tensor] = None,
    ) -> Tensor:
        adj = self._strip_self_loops(adj)
        positive_pair_counts = self._positive_pair_counts(adj)

        intra_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_a))
        inter_sim = self.tau_scaling(self.similarity_matrix(emb_a, emb_b))

        if sample_mask is not None:
            intra_sim = intra_sim * sample_mask
            inter_sim = inter_sim * sample_mask

        numerator = (
            inter_sim.diag() + intra_sim.mul(adj).sum(1) + inter_sim.mul(adj).sum(1)
        )
        denominator = intra_sim.sum(1) + inter_sim.sum(1) - intra_sim.diag()

        return -torch.log((numerator / denominator) / positive_pair_counts)

    def contrastive_loss(
        self,
        emb_a: Tensor,
        emb_b: Tensor,
        adj: Tensor,
        sample_mask: Optional[Tensor] = None,
        mean: bool = True,
    ) -> Tensor:
        per_node = (
            self._neighbor_contrastive_loss(emb_a, emb_b, adj, sample_mask)
            + self._neighbor_contrastive_loss(emb_b, emb_a, adj, sample_mask)
        ) * 0.5
        return self._reduce(per_node, mean)

#### 3.3.4 Utility Functions:
*   **Graph Augmentation**: Implements feature dropping and edge dropout to robustify the model against noise.
*   **Clustering**: Applies `MCLust`, `Leiden`, or `Louvain` algorithms on the learned embeddings to assign domain labels, followed by a refinement step to smooth spatial boundaries.

In [ ]:
def drop_feature(node_features: Tensor, drop_prob: float) -> Tensor:
    """Randomly zeros out feature dimensions with probability `drop_prob`."""
    drop_mask = (
        torch.empty(node_features.size(1), device=torch.device("cpu")).uniform_(0, 1)
        < drop_prob
    )
    node_features = node_features.clone()
    node_features[:, drop_mask] = 0
    return node_features


def filter_adj(
    row: Tensor, col: Tensor, edge_attr: OptTensor, keep_mask: Tensor
) -> Tuple[Tensor, Tensor, OptTensor]:
    filtered_attr = None if edge_attr is None else edge_attr[keep_mask]
    return row[keep_mask], col[keep_mask], filtered_attr


def dropout_adj(
    edge_index: Tensor,
    edge_attr: Tensor,
    force_undirected: bool = False,
    num_nodes: Optional[int] = None,
    training: bool = True,
) -> Tuple[Tensor, Tensor]:
    if not training:
        return edge_index, edge_attr

    row, col = edge_index

    if force_undirected:
        upper_tri_mask = row <= col
        row, col, edge_attr = (
            row[upper_tri_mask],
            col[upper_tri_mask],
            edge_attr[upper_tri_mask],
        )

    keep_mask = torch.rand(
        edge_attr.size(0), device=torch.device("cpu")
    ) >= edge_attr.to("cpu")
    row, col, edge_attr = filter_adj(row, col, edge_attr, keep_mask)

    if force_undirected:
        edge_index = torch.stack([torch.cat([row, col]), torch.cat([col, row])], dim=0)
    else:
        edge_index = torch.stack([row, col], dim=0)
    
    return edge_index, edge_attr


def multiple_dropout_average(
    edge_index: Tensor,
    edge_attr: Tensor,
    num_trials: int = 10,
    force_undirected: bool = False,
    num_nodes: Optional[int] = None,
    threshold_ratio: float = 0.5,
    training: bool = True,
    device: str = "cuda",
) -> Tuple[Tensor, Tensor]:
    if not training:
        return edge_index, edge_attr

    if num_nodes is None:
        num_nodes = int(edge_index.max().item()) + 1

    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)

    use_simulation = False
    if use_simulation:
        edge_count = torch.zeros(
            (num_nodes, num_nodes), dtype=torch.int32, device=device
        )
        for _ in range(num_trials):
            dropped_edge_index, _ = dropout_adj(edge_index, edge_attr, force_undirected)
            dropped_edge_index = dropped_edge_index.to(device)
            src, dst = dropped_edge_index
            edge_count[src, dst] += 1
            if force_undirected:
                edge_count[dst, src] += 1
        threshold = int(num_trials * threshold_ratio)
        final_edge_index = (edge_count >= threshold).nonzero().t().contiguous()
    else:
        final_edge_index, _ = dropout_adj(edge_index, edge_attr, force_undirected)

    return final_edge_index, edge_attr


def random_dropout_adj(
    edge_index: Tensor,
    edge_attr: OptTensor = None,
    p: float = 0.5,
    force_undirected: bool = False,
    num_nodes: Optional[int] = None,
    training: bool = True,
) -> Tuple[Tensor, OptTensor]:
    if not 0.0 <= p <= 1.0:
        raise ValueError(f"Dropout probability has to be between 0 and 1 (got {p})")

    if not training or p == 0.0:
        return edge_index, edge_attr

    row, col = edge_index
    keep_mask = torch.rand(row.size(0), device=torch.device("cpu")) >= p

    if force_undirected:
        keep_mask[row > col] = False

    row, col, edge_attr = filter_adj(row, col, edge_attr, keep_mask)

    if force_undirected:
        edge_index = torch.stack([torch.cat([row, col]), torch.cat([col, row])], dim=0)
        if edge_attr is not None:
            edge_attr = torch.cat([edge_attr, edge_attr], dim=0)
    else:
        edge_index = torch.stack([row, col], dim=0)

    return edge_index, edge_attr

In [ ]:
# Discriminator for DGI-style loss (not used in main training loop)
class Discriminator(nn.Module):
    _NUM_LAYERS = 1
    _HIDDEN_DIM = 64
    _DROPOUT = 0.2
    _INPUT_DROPOUT = 0.1

    def __init__(self, input_dim: int):
        super().__init__()

        layers: list[nn.Module] = [nn.Dropout(self._INPUT_DROPOUT)]
        for i in range(self._NUM_LAYERS + 1):
            layer_in = input_dim if i == 0 else self._HIDDEN_DIM
            layer_out = 1 if i == self._NUM_LAYERS else self._HIDDEN_DIM
            layers.append(nn.Linear(layer_in, layer_out))
            if i < self._NUM_LAYERS:
                layers += [nn.ReLU(), nn.Dropout(self._DROPOUT)]

        self.layers = nn.Sequential(*layers)

    def forward(self, node_features: Tensor) -> Tensor:
        return self.layers(node_features).view(-1)

### 3.4 Clustering and Post-Processing
Once the STAGM model generates the latent embeddings, the `clustering` module assigns discrete domain labels to each spatial spot.
**Methodology:**
1.  **Algorithm Selection**: Supports `MCLust`, `Leiden`, and `Louvain` algorithms.
    *   `MCLust`: Uses Gaussian Mixture Models (GMM) to fit the distribution of embeddings.
    *   `Leiden/Louvain`: Graph-based community detection algorithms optimized for modularity. The resolution parameter is automatically tuned (via `search_res`) to match a target cluster count.
2.  **Resolution Optimization**: The pipeline performs a grid search over the resolution parameter to find the optimal setting that yields the desired number of clusters, ensuring biological relevance.
3.  **Spatial Refinement**: A post-processing step (`refine_label`) enforces spatial consistency.
    *   **Mechanism**: For each spot, the algorithm identifies its $k$-nearest spatial neighbors and performs a majority voting step.
    *   **Goal**: This smooths out isolated misclassifications (salt-and-pepper noise) and ensures that contiguous regions of tissue are assigned the same domain label.
4.  **Evaluation**: Finally, the pipeline computes Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI) against ground truth labels, as well as Silhouette and Davies-Bouldin scores to assess intrinsic clustering quality.


In [ ]:
def clustering(
    adata,
    n_clusters=7,
    radius=50,
    key="emb",
    method="mclust",
    start=0.1,
    end=3.0,
    increment=0.01,
    refinement=False,
):
    """
    Spatial clustering based the learned representation.

    Parameters
    ----------
    adata : anndata
        AnnData object of scanpy package.
    n_clusters : int, optional
        The number of clusters. The default is 7.
    radius : int, optional
        The number of neighbors considered during refinement. The default is 50.
    key : string, optional
        The key of the learned representation in adata.obsm. The default is 'emb'.
    method : string, optional
        The tool for clustering. Supported tools include 'mclust', 'leiden', and 'louvain'. The default is 'mclust'.
    start : float
        The start value for searching. The default is 0.1.
    end : float
        The end value for searching. The default is 3.0.
    increment : float
        The step size to increase. The default is 0.01.
    refinement : bool, optional
        Refine the predicted labels or not. The default is False.

    Returns
    -------
    None.

    """
    if method == "mclust":
        adata = mclust_R(adata, used_obsm=key, num_cluster=n_clusters)
        adata.obs["domain"] = adata.obs["mclust"]

    elif method in ("leiden", "louvain"):
        res = search_res(
            radius,
            adata,
            n_clusters,
            use_rep=key,
            method=method,
            start=start,
            end=end,
            increment=increment,
        )

        if method == "leiden":
            sc.tl.leiden(adata, random_state=0, resolution=res)

        else:
            sc.tl.louvain(adata, random_state=0, resolution=res)

        adata.obs["domain"] = adata.obs[method]

    if refinement:
        new_type = refine_label(adata, radius, key="domain")
        adata.obs["domain"] = new_type

In [ ]:
def refine_label(adata, radius=50, key="label"):
    n_neigh = radius
    old_type = adata.obs[key].values

    # Calculate pairwise euclidean distances between spatial positions
    position = adata.obsm["spatial"]
    distance = ot.dist(position, position, metric="euclidean")

    n_cell = distance.shape[0]
    new_type = []
    for i in range(n_cell):
        index = distance[i, :].argsort()
        neigh_type = [old_type[index[j]] for j in range(1, n_neigh + 1)]
        new_type.append(max(neigh_type, key=neigh_type.count))

    return [str(i) for i in new_type]


In [ ]:
# Search resolution for Leiden/Louvain clustering

from sklearn import metrics


def search_res(
    radius,
    adata,
    n_clusters,
    method="leiden",
    use_rep="norm_emb",
    start=0.1,
    end=3.0,
    increment=0.01,
):
    """
    Searching corresponding resolution according to given cluster number

    Parameters
    ----------
    adata : anndata
        AnnData object of spatial data.
    n_clusters : int
        Targetting number of clusters.
    method : string
        Tool for clustering. Supported tools include 'leiden' and 'louvain'. The default is 'leiden'.
    use_rep : string
        The indicated representation for clustering.
    start : float
        The start value for searching.
    end : float
        The end value for searching.
    increment : float
        The step size to increase.

    Returns
    -------
    res : float
        Resolution.

    """

    def _cluster(resolution):
        """Run the chosen clustering method and return the unique cluster count."""
        if method == "leiden":
            sc.tl.leiden(adata, random_state=0, resolution=resolution)
            return len(adata.obs["leiden"].unique())
        else:
            sc.tl.louvain(adata, random_state=0, resolution=resolution)
            return len(adata.obs["louvain"].unique())

    print("Searching resolution...")
    sc.pp.neighbors(adata, n_neighbors=20, use_rep=use_rep)

    # Coarsely adjust `end` so the upper-bound cluster count is n_clusters + 2
    count_unique = _cluster(end)
    while count_unique > n_clusters + 2:
        print(f"Cluster count {count_unique} is too large, adjusting end downward...")
        end -= 0.1
        count_unique = _cluster(end)
    while count_unique < n_clusters + 2:
        print(f"Cluster count {count_unique} is too small, adjusting end upward...")
        end += 0.1
        count_unique = _cluster(end)

    # Fine-grained search over [start, end)
    ress = []
    found = False
    for res in sorted(np.arange(start, end, increment), reverse=True):
        count_unique = _cluster(res)
        print(f"resolution={res:.4f}, cluster number={count_unique}")

        if count_unique == n_clusters:
            new_type = refine_label(adata, radius, key="leiden")
            adata.obs["leiden"] = new_type
            ARI = metrics.adjusted_rand_score(
                adata.obs["leiden"], adata.obs["ground_truth"]
            )
            adata.uns["ARI"] = ARI
            ress.append((res, ARI))
            print(f"ARI: {ARI:.4f}")

        if count_unique == n_clusters - 2:
            found = True
            best = max(ress, key=lambda x: x[1])
            print(f"Best resolution found: {best}")
            break

    assert found, "Resolution not found. Please try a bigger range or a smaller step."

    return best[0]


---
<a name="4.-Run-an-experiment"></a>
## 4. Run an Experiment

### 4.1 Configuration and Reproducibility
This section defines the hyperparameters for the STAGM model, including network depth, hidden dimensions, dropout rates, and Mamba-specific parameters ($d_{state}, d_{conv}$). It also initializes random seeds for PyTorch, NumPy, and the random module to ensure experimental reproducibility.

In [ ]:
# Dataclass for the configuration of the model
@dataclass
class model_config:
    seed: int
    learning_rate: float
    num_hidden: int
    num_proj_hidden: int
    activation: str
    base_model: str
    num_layers: int
    drop_feature_rate_1: float
    drop_feature_rate_2: float
    tau: int
    num_epochs: int
    weight_decay: float
    num_clusters: int
    num_gene: int
    num_neigh: int
    k: int
    dropout: float
    order_by_degree: bool
    shuffle_ind: int
    d_state: int
    d_conv: int


# Dataclass for the configuration of the dataset
@dataclass
class dataset_config:
    dataset: str
    slide: str
    config_file: str
    label: bool

In [ ]:
# Initialize the model configuration with the specified parameters
model_config = model_config(
    seed=39788,
    learning_rate=0.0005,
    num_hidden=64,
    num_proj_hidden=64,
    activation="prelu",
    base_model="GCNConv",
    num_layers=1,
    drop_feature_rate_1=0.1,
    drop_feature_rate_2=0.2,
    tau=35,
    num_epochs=300,
    weight_decay=1e-05,
    num_clusters=7,
    num_gene=3000,
    num_neigh=5,
    k=80,
    dropout=0.0,
    order_by_degree=True,
    shuffle_ind=0,
    d_state=16,
    d_conv=4,
)


# Initialize the dataset configuration with the specified parameters
args = dataset_config(
    dataset="DLPFC", slide="151673", config_file="train_img_config.yaml", label=True
)


# Define the base "/Dataset" folder and the path to the specific dataset and slide
data_path = Path("./Dataset")
data_path = data_path / args.dataset / args.slide
data_path

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(model_config.seed)
np.random.seed(model_config.seed)


# Set torch to deterministic for experiment reproducibility
if torch.cuda.is_available():
    torch.cuda.manual_seed(model_config.seed)
    torch.cuda.manual_seed_all(model_config.seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
random.seed(model_config.seed)

torch.use_deterministic_algorithms(True)


# Configure cuBLAS memory allocation to 4096MB 8 workspaces
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

### 4.2 Data Initialization
The experiment loads the specific dataset using the configuration defined above. The data undergoes the preprocessing pipeline described in Section 3.1, resulting in a fully prepared `AnnData` object containing graph structures, multimodal features, and edge weights.

In [ ]:
# Load the AnnData object using the specified parameters
adata: AnnData = LoadSingle10xAdata(
    path=str(data_path),
    n_neighbors=model_config.num_neigh,
    n_top_genes=model_config.num_gene,
    image_emb=True,
    label=args.label,
).run()

### 4.3 Model Instantiation
The STAGM model is initialized with the prepared data. The `single=False` flag indicates the use of the **Multi-View** configuration, which leverages image embeddings to generate pseudo-labels for a more informed contrastive learning objective.


In [ ]:
# Initialize the STAGM model with the specified parameters and assign the loaded AnnData object to it
stagm = STAGM(args=args, config=model_config, single=False)
stagm.adata = adata

### 4.4 Training Phase
The model is trained using an Adam optimizer to minimize the contrastive loss. During this phase:
1.  **Data Augmentation**: The graph and features are stochastically dropped to simulate different views of the data.
2.  **Forward Pass**: The encoder generates embeddings for both augmented views.
3.  **Optimization**: The gradient is computed based on the neighbor-aware contrastive loss and backpropagated to update model weights.
The training progress is tracked via a loss curve.


In [ ]:
# Train the STAGM model
stagm.train()

### 4.5 Evaluation and Diagnostics
After training, the model generates final embeddings for all spots. This cell performs:
1.  **Diagnostics**: Reports model parameters, memory footprint, and training efficiency.
2.  **Embedding Statistics**: Analyzes the distribution and norms of the learned representations to ensure stability.

In [ ]:
# Evaluate the STAGM model
stagm.eva()

### 4.6 Clustering and Metric Calculation
The learned embeddings are clustered using the selected algorithm.
*   **Refinement**: A spatial smoothing step ensures that adjacent spots are more likely to share the same cluster label.
*   **Performance Metrics**:
    *   **ARI (Adjusted Rand Index)**: Measures similarity between predicted domains and ground truth labels.
    *   **NMI (Normalized Mutual Information)**: Quantifies the information shared between the clustering and true labels.
    *   **Silhouette Score & Davies-Bouldin Index**: Evaluate clustering quality based on the embedding geometry.


In [ ]:
# Perform clustering using the trained STAGM model
stagm.cluster(args.label)

In [ ]:
# Draw the spatial visualization of the clusters
stagm.draw_spatial()